In [142]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [143]:
INPUT_FILE = 'data/FINAL_COMPLETE_HateCrimes.csv'
OUTPUT_FILE = 'data/HateCrimes_Clean.csv'

### Load Data

In [144]:
try:
    df = pd.read_csv(INPUT_FILE, low_memory=False)
    print(f"Loaded {len(df):,} records with {len(df.columns)} columns")
    initial_rows = len(df)
except Exception as e:
    print(f"Error loading file: {e}")
    exit(1)

Loaded 46,675 records with 27 columns


In [145]:
df.head(10)

,incident_date,year,month,day_of_week,quarter,state_name,region,division,city,location,...,bias_motivation,total_victims,adult_victims,juvenile_victims,total_offenders,adult_offenders,juvenile_offenders,victim_types,offender_race,offender_ethnicity
0,2021-01-01,2021,1,Friday,Q1,Alabama,South,East South Central,CROSSVILLE,Residence/Home,...,Anti-White,1,1,0,1,1,0,Individual,Black or African American,Not Hispanic or Latino
1,2021-01-01,2021,1,Friday,Q1,Arizona,South,East South Central,LIVINGSTON,Residence/Home,...,Anti-Church of Jesus Christ (LDS),1,1,0,1,1,0,Individual,White,Not Hispanic or Latino
2,2021-01-01,2021,1,Friday,Q1,California,South,East South Central,RAINBOW CITY,Residence/Home,...,Anti-Black or African American,2,2,0,1,1,0,Individual,White,Not Hispanic or Latino
3,2021-01-01,2021,1,Friday,Q1,California,South,East South Central,UNIONTOWN,Highway/Road/Alley,...,Anti-Black or African American,1,1,0,1,1,0,Individual,White,Not Hispanic or Latino
4,2021-01-01,2021,1,Friday,Q1,California,South,East South Central,WEST BLOCTON,Restaurant,...,Anti-Gay (Male),1,1,0,1,1,0,Individual,Unknown,Hispanic or Latino
5,2021-01-01,2021,1,Friday,Q1,California,West,Pacific,BARROW,Park/Playground,...,Anti-Jewish,0,0,0,0,0,0,"Business, Government",Unknown,NaN
6,2021-01-01,2021,1,Friday,Q1,Colorado,West,Mountain,Fort McDowell,"Specialty Store (TV, Fur, Etc.)",...,Anti-Arab,3,3,0,1,1,0,Individual,White,Not Hispanic or Latino
7,2021-01-01,2021,1,Friday,Q1,Colorado,West,Mountain,GILA BEND,Parking Lot/Garage,...,Anti-Hispanic or Latino,1,1,0,1,1,0,Individual,American Indian/Alaska Native,Unknown
8,2021-01-01,2021,1,Friday,Q1,Colorado,West,Mountain,KINGMAN,Highway/Road/Alley,...,Anti-Black or African American,1,1,0,1,1,0,Individual,White,Hispanic or Latino
9,2021-01-01,2021,1,Friday,Q1,Unknown,West,Mountain,NaN,Church/Synagogue/Temple,...,Anti-Black or African American,0,0,0,0,0,0,Religious Organization,Unknown,Unknown


### Data Normalization

In [146]:
# Convert incident_date to datetime
df['incident_date'] = pd.to_datetime(df['incident_date'], format='%Y-%m-%d', errors='coerce')
print(f"Converted incident_date to datetime")
print(f"Date range: {df['incident_date'].min()} to {df['incident_date'].max()}")

# Convert numeric columns
numeric_cols = [
    'year', 'month', 'population', 
    'total_victims', 'adult_victims', 'juvenile_victims',
    'total_offenders', 'adult_offenders', 'juvenile_offenders'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"Converted {len(numeric_cols)} columns to numeric")

# Clean string columns (strip whitespace)
string_cols = df.select_dtypes(include=['object']).columns
for col in string_cols:
    df[col] = df[col].astype(str).str.strip()

print(f"Cleaned {len(string_cols)} string columns")

Converted incident_date to datetime
Date range: 2021-01-01 00:00:00 to 2024-12-31 00:00:00
Converted 9 columns to numeric
Cleaned 17 string columns


### Handle Missing Value and Placeholders

In [147]:
# Replace string placeholders with NaN
placeholder_strings = ['nan', 'NaN', 'None', '', 'U', 'Unknown', 'UNKNOWN', 
                       'N/A', 'NA', '#N/A']

for col in string_cols:
    df[col] = df[col].replace(placeholder_strings, np.nan)

replaced_strings = sum((df[col].isin(placeholder_strings)).sum() for col in string_cols)
print(f"Replaced string placeholders with NaN")

# Replace numeric placeholders (0, -1) with NaN where appropriate
# For victim/offender counts: 0 is actually valid for property crimes
# But "Unknown" offender race should be NaN
victim_offender_cols = ['total_victims', 'adult_victims', 'juvenile_victims',
                        'total_offenders', 'adult_offenders', 'juvenile_offenders']

# Replace -1 with NaN (invalid)
for col in victim_offender_cols:
    if col in df.columns:
        negative_mask = df[col] < 0
        neg_count = negative_mask.sum()
        if neg_count > 0:
            df.loc[negative_mask, col] = np.nan
            print(f"Fixed {neg_count} negative values in {col}")

# Handle population = 0 (likely missing, not actual zero)
if 'population' in df.columns:
    zero_pop = (df['population'] == 0).sum()
    df.loc[df['population'] == 0, 'population'] = np.nan
    print(f"Converted {zero_pop} zero population values to NaN")

# Clean specific categorical fields
# "Unknown" in offender_race and offender_ethnicity should be NaN
if 'offender_race' in df.columns:
    unknown_race = df['offender_race'].isin(['Unknown', 'UNKNOWN', 'U']).sum()
    df['offender_race'] = df['offender_race'].replace(['Unknown', 'UNKNOWN', 'U'], np.nan)
    print(f"Converted {unknown_race} Unknown offender_race to NaN")

if 'offender_ethnicity' in df.columns:
    unknown_eth = df['offender_ethnicity'].isin(['Unknown', 'UNKNOWN', 'U']).sum()
    df['offender_ethnicity'] = df['offender_ethnicity'].replace(['Unknown', 'UNKNOWN', 'U'], np.nan)
    print(f"Converted {unknown_eth} Unknown offender_ethnicity to NaN")

Replaced string placeholders with NaN
Converted 15653 zero population values to NaN
Converted 0 Unknown offender_race to NaN
Converted 0 Unknown offender_ethnicity to NaN


### Categorical Normalization

In [148]:
# Standardize offender_race
if 'offender_race' in df.columns:
    race_mapping = {
        'W': 'White', 'WHITE': 'White', 'white': 'White',
        'B': 'Black or African American', 'BLACK': 'Black or African American',
        'black': 'Black or African American',
        'A': 'Asian', 'ASIAN': 'Asian', 'asian': 'Asian',
        'I': 'American Indian/Alaska Native',
        'P': 'Native Hawaiian/Other Pacific Islander',
        'M': 'Multiple Races', 'Group of Multiple Races': 'Multiple Races'
    }
    df['offender_race'] = df['offender_race'].replace(race_mapping)
    print(f"Normalized offender_race: {df['offender_race'].nunique()} categories")

# Standardize offender_ethnicity
if 'offender_ethnicity' in df.columns:
    ethnicity_mapping = {
        'H': 'Hispanic or Latino',
        'HISPANIC': 'Hispanic or Latino',
        'N': 'Not Hispanic or Latino',
        'NOT HISPANIC': 'Not Hispanic or Latino'
    }
    df['offender_ethnicity'] = df['offender_ethnicity'].replace(ethnicity_mapping)
    print(f"Normalized offender_ethnicity: {df['offender_ethnicity'].nunique()} categories")

# Normalize bias_motivation (consistent capitalization)
if 'bias_motivation' in df.columns:
    # Keep as-is since they're already properly formatted
    # Just ensure consistency
    df['bias_motivation'] = df['bias_motivation'].str.strip()
    print(f"Verified bias_motivation: {df['bias_motivation'].nunique()} types")

# Normalize geographic names (title case)
geo_cols = ['state_name', 'region', 'division', 'city']
for col in geo_cols:
    if col in df.columns:
        # Only title case if not all caps or mixed
        df[col] = df[col].str.title()

print(f"Normalized {len(geo_cols)} geographic columns")

# Standardize bias_category
if 'bias_category' in df.columns:
    bias_mapping = {
        'Race/Ethnicity': 'Race/Ethnicity',
        'race/ethnicity': 'Race/Ethnicity',
        'RACE': 'Race/Ethnicity',
        'Religion': 'Religion',
        'religion': 'Religion',
        'Sexual Orientation': 'Sexual Orientation',
        'sexual orientation': 'Sexual Orientation',
        'Gender Identity': 'Gender Identity',
        'gender identity': 'Gender Identity',
        'Disability': 'Disability',
        'disability': 'Disability',
        'Gender': 'Gender',
        'gender': 'Gender'
    }
    df['bias_category'] = df['bias_category'].replace(bias_mapping)
    print(f"Normalized bias_category: {df['bias_category'].nunique()} groups")

Normalized offender_race: 6 categories
Normalized offender_ethnicity: 2 categories
Verified bias_motivation: 34 types
Normalized 4 geographic columns
Normalized bias_category: 6 groups


### Remove Duplicates

In [149]:
# Check for complete duplicates
before_dup = len(df)
df = df.drop_duplicates(keep='first')
complete_dups = before_dup - len(df)
print(f"Removed {complete_dups} complete duplicate records")

Removed 464 complete duplicate records


### Data Validation and Logic Checks

In [150]:
issues_found = []
# (đoạn này bỏ được check cột total có bằng các cột component cộng lại không thôi)
# Check victim totals consistency (đoạn này bỏ được)
if all(c in df.columns for c in ['adult_victims', 'juvenile_victims', 'total_victims']):
    df['computed_victims'] = df['adult_victims'].fillna(0) + df['juvenile_victims'].fillna(0)
    # Allow some tolerance (rounding errors)
    mismatch = abs(df['computed_victims'] - df['total_victims'].fillna(0)) > 1
    mismatch_count = mismatch.sum()
    
    if mismatch_count > 0:
        # Fix mismatches by using computed value
        df.loc[mismatch, 'total_victims'] = df.loc[mismatch, 'computed_victims']
        issues_found.append(f"Fixed {mismatch_count} victim count mismatches")
    
    df = df.drop('computed_victims', axis=1)

# Check offender totals consistency
if all(c in df.columns for c in ['adult_offenders', 'juvenile_offenders', 'total_offenders']):
    df['computed_offenders'] = df['adult_offenders'].fillna(0) + df['juvenile_offenders'].fillna(0)
    mismatch = abs(df['computed_offenders'] - df['total_offenders'].fillna(0)) > 1
    mismatch_count = mismatch.sum()
    
    if mismatch_count > 0:
        df.loc[mismatch, 'total_offenders'] = df.loc[mismatch, 'computed_offenders']
        issues_found.append(f"Fixed {mismatch_count} offender count mismatches")
    
    df = df.drop('computed_offenders', axis=1)

# Validate date range
if 'incident_date' in df.columns:
    future_dates = df['incident_date'] > pd.Timestamp.now()
    future_count = future_dates.sum()
    if future_count > 0:
        df.loc[future_dates, 'incident_date'] = pd.NaT
        issues_found.append(f"Removed {future_count} future dates")
    
    # Check for very old dates (before 2000)
    old_dates = df['incident_date'] < pd.Timestamp('2000-01-01')
    old_count = old_dates.sum()
    if old_count > 0:
        issues_found.append(f"Found {old_count} incidents before 2000 (verify)")

# Validate year-month consistency
if all(c in df.columns for c in ['incident_date', 'year', 'month']):
    df['year_check'] = df['incident_date'].dt.year
    df['month_check'] = df['incident_date'].dt.month
    
    year_mismatch = (df['year'] != df['year_check']).sum()
    month_mismatch = (df['month'] != df['month_check']).sum()
    
    if year_mismatch > 0 or month_mismatch > 0:
        # Fix by using incident_date as source of truth
        df['year'] = df['incident_date'].dt.year
        df['month'] = df['incident_date'].dt.month
        issues_found.append(f"Fixed {year_mismatch} year and {month_mismatch} month mismatches")
    
    df = df.drop(['year_check', 'month_check'], axis=1)

# Check for critical missing values
critical_fields = ['incident_date', 'bias_category', 'offense_severity']
for field in critical_fields:
    if field in df.columns:
        missing_count = df[field].isna().sum()
        missing_pct = (missing_count / len(df)) * 100
        if missing_count > 0:
            issues_found.append(f"{missing_count} ({missing_pct:.2f}%) records missing {field}")

if issues_found:
    print("   Issues Found & Fixed:")
    for issue in issues_found:
        print(issue)
else:
    print("No critical validation issues detected")

   Issues Found & Fixed:
138 (0.30%) records missing offense_severity


### Feature Transformation and Engineering

In [151]:
# Ensure all temporal features are present and correct
if 'incident_date' in df.columns:
    df['year'] = df['incident_date'].dt.year
    df['month'] = df['incident_date'].dt.month
    df['day_of_week'] = df['incident_date'].dt.day_name()
    df['quarter'] = 'Q' + df['incident_date'].dt.quarter.astype(str)
    df['is_weekend'] = df['incident_date'].dt.dayofweek.isin([5, 6])
    df['day_of_year'] = df['incident_date'].dt.dayofyear
    print(f"Updated temporal features (year, month, quarter, etc.)")

# Create victim-to-offender ratio
if 'total_victims' in df.columns and 'total_offenders' in df.columns:
    df['victim_offender_ratio'] = df['total_victims'] / df['total_offenders']
    # Replace infinity with NaN
    df['victim_offender_ratio'] = df['victim_offender_ratio'].replace([np.inf, -np.inf], np.nan)
    print(f"Created victim_offender_ratio")

# Create severity score
if 'offense_severity' in df.columns:
    severity_mapping = {
        'Violent': 3,
        'Non-Violent': 1,
        'Unknown': np.nan
    }
    df['severity_score'] = df['offense_severity'].map(severity_mapping)
    print(f"Created severity_score (3=Violent, 1=Non-Violent)")

# Create population category
if 'population' in df.columns:
    df['population_category'] = pd.cut(
        df['population'],
        bins=[-1, 2500, 10000, 50000, 100000, 500000, float('inf')],
        labels=['Very Small', 'Small', 'Medium', 'Large', 'Very Large', 'Major City']
    )
    print(f"Created population_category")

# Create bias group simplified
if 'bias_category' in df.columns:
    # Already exists, but ensure clean version
    df['bias_group'] = df['bias_category'].copy()
    print(f"Created bias_group (alias for bias_category)")

# Create hate crime severity index (composite score)
if all(c in df.columns for c in ['severity_score', 'total_victims']):
    df['hate_crime_index'] = df['severity_score'] * np.log1p(df['total_victims'].fillna(1))
    print(f"Created hate_crime_index (composite severity measure)")

# Create month name
if 'month' in df.columns:
    month_names = {
        1: 'January', 2: 'February', 3: 'March', 4: 'April',
        5: 'May', 6: 'June', 7: 'July', 8: 'August',
        9: 'September', 10: 'October', 11: 'November', 12: 'December'
    }
    df['month_name'] = df['month'].map(month_names)
    print(f"Created month_name")

# Create region codes
if 'region' in df.columns:
    region_codes = {
        'Northeast': 1, 'North Central': 2, 'South': 3, 'West': 4, 'Possessions': 0
    }
    df['region_code'] = df['region'].map(region_codes)
    print(f"Created region_code")

Updated temporal features (year, month, quarter, etc.)
Created victim_offender_ratio
Created severity_score (3=Violent, 1=Non-Violent)
Created population_category
Created bias_group (alias for bias_category)
Created hate_crime_index (composite severity measure)
Created month_name
Created region_code


### Final Cleaning and Quality Control

In [152]:
# Remove rows with excessive missing values (>70% of columns missing)
threshold = len(df.columns) * 0.3  # Keep if at least 30% filled
before_removal = len(df)
df = df.dropna(thresh=threshold)
removed = before_removal - len(df)
print(f"Removed {removed} records with excessive missing data")

# Remove records missing critical fields
critical_required = ['incident_date', 'bias_category']
before_critical = len(df)
df = df.dropna(subset=critical_required)
removed_critical = before_critical - len(df)
print(f"Removed {removed_critical} records missing critical fields")

# Reset index
df = df.reset_index(drop=True)

# Sort by date and state
df = df.sort_values(['incident_date', 'state_name', 'city'], na_position='last')

# Reorder columns logically
preferred_order = [
    'incident_date', 'year', 'month', 'month_name', 'quarter', 'day_of_week', 
    'is_weekend', 'day_of_year',
    'state_name', 'region', 'region_code', 'division', 'city', 
    'agency_name', 'agency_type', 'population', 'population_category', 'population_group',
    'location', 'offense_description', 'offense_severity', 'severity_score',
    'bias_category', 'bias_group', 'bias_motivation',
    'total_victims', 'adult_victims', 'juvenile_victims',
    'total_offenders', 'adult_offenders', 'juvenile_offenders',
    'victim_offender_ratio', 'hate_crime_index',
    'victim_types', 'offender_race', 'offender_ethnicity'
]

# Reorder columns that exist
existing_order = [c for c in preferred_order if c in df.columns]
other_cols = [c for c in df.columns if c not in existing_order]
df = df[existing_order + other_cols]

print(f"Reordered columns for better usability")

Removed 0 records with excessive missing data
Removed 0 records missing critical fields
Reordered columns for better usability


### Data Quality Report

In [153]:
print(f"\n FINAL DATASET SUMMARY")
print(f"   Total Records: {len(df):,} (started with {initial_rows:,})")
print(f"   Records Removed: {initial_rows - len(df):,} ({((initial_rows - len(df))/initial_rows*100):.2f}%)")
print(f"   Total Columns: {len(df.columns)}")
if 'incident_date' in df.columns and df['incident_date'].notna().any():
    min_date = df['incident_date'].min()
    max_date = df['incident_date'].max()
    if pd.notna(min_date) and pd.notna(max_date):
        print(f"Date Range: {min_date.strftime('%Y-%m-%d')} to {max_date.strftime('%Y-%m-%d')}")
if 'total_victims' in df.columns:
    print(f"   Total Victims: {df['total_victims'].sum():,.0f}")
if 'total_offenders' in df.columns:
    print(f"   Total Offenders: {df['total_offenders'].sum():,.0f}")

print(f"\n GEOGRAPHIC COVERAGE")
if 'state_name' in df.columns:
    print(f"   States/Territories: {df['state_name'].nunique()}")
    print(f"   Top 5 States by Incidents:")
    for state, count in df['state_name'].value_counts().head(5).items():
        print(f"      {state:20s}: {count:6,} incidents")

print(f"\n BIAS CATEGORIES")
if 'bias_category' in df.columns:
    for category, count in df['bias_category'].value_counts().items():
        pct = (count / len(df)) * 100
        print(f"   {category:25s}: {count:6,} ({pct:5.2f}%)")

print(f"\n MISSING DATA SUMMARY (Top 10 columns)")
missing_summary = df.isna().sum().sort_values(ascending=False)
missing_summary = missing_summary[missing_summary > 0].head(10)
for col, count in missing_summary.items():
    pct = (count / len(df)) * 100
    print(f"   {col:30s}: {count:6,} ({pct:5.2f}%)")


 FINAL DATASET SUMMARY
   Total Records: 46,211 (started with 46,675)
   Records Removed: 464 (0.99%)
   Total Columns: 36
Date Range: 2021-01-01 to 2024-12-31
   Total Victims: 44,785
   Total Offenders: 33,558

 GEOGRAPHIC COVERAGE
   States/Territories: 52
   Top 5 States by Incidents:
      California          :  7,781 incidents
      New Jersey          :  4,321 incidents
      New York            :  3,489 incidents
      Washington          :  2,149 incidents
      Texas               :  2,048 incidents

 BIAS CATEGORIES
   Race/Ethnicity           : 26,129 (56.54%)
   Religion                 :  9,235 (19.98%)
   Sexual Orientation       :  7,961 (17.23%)
   Gender Identity          :  1,796 ( 3.89%)
   Disability               :    697 ( 1.51%)
   Gender                   :    393 ( 0.85%)

 MISSING DATA SUMMARY (Top 10 columns)
   offender_ethnicity            : 30,690 (66.41%)
   offender_race                 : 19,284 (41.73%)
   victim_offender_ratio         : 18,964 (41.04

### Export Cleaned Data

In [154]:
try:
    # Export main clean file
    df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
    print(f" Exported: {OUTPUT_FILE}")
    
    # Export summary statistics
    summary_file = OUTPUT_FILE.replace('.csv', '_Summary.txt')
    with open(summary_file, 'w') as f:
        f.write("FBI HATE CRIME DATA - CLEANING SUMMARY\n")
        f.write("="*80 + "\n\n")
        f.write(f"Original Records: {initial_rows:,}\n")
        f.write(f"Final Records: {len(df):,}\n")
        f.write(f"Records Removed: {initial_rows - len(df):,}\n")
        f.write(f"Date Range: {df['incident_date'].min()} to {df['incident_date'].max()}\n")
        f.write(f"\nColumns: {len(df.columns)}\n")
        f.write(f"\nMissing Data Profile:\n")
        for col, count in missing_summary.items():
            f.write(f"  {col}: {count:,} ({count/len(df)*100:.2f}%)\n")
    
    print(f" Exported summary: {summary_file}")
    
except Exception as e:
    print(f" Error exporting files: {e}")

 Exported: data/HateCrimes_Clean.csv
 Exported summary: data/HateCrimes_Clean_Summary.txt


In [155]:
print(" DATA CLEANING COMPLETED SUCCESSFULLY!")
print("="*80)
print(f"\n Dataset cleaned and ready for analysis")
print(f" {len(df):,} high-quality records")
print(f" {len(df.columns)} features including engineered variables")
print(f" Temporal range: {df['year'].min():.0f} - {df['year'].max():.0f}")
print(f"\n Output: {OUTPUT_FILE}")

 DATA CLEANING COMPLETED SUCCESSFULLY!

 Dataset cleaned and ready for analysis
 46,211 high-quality records
 36 features including engineered variables
 Temporal range: 2021 - 2024

 Output: data/HateCrimes_Clean.csv
